# Spatial Analysis of the Physics Residual (∇²T̂)

**Goal:** understand *where* and *how much* the trained ThermalNet violates the PDE constraint ∇²T = 0 at every node, for different cases.

We explore:
1. **Single-case map** — 4-panel 3D scatter: FEM T | predicted T | |error| | |∇²T̂|
2. **Boundary profile** — Laplacian residual vs x-position (does it peak near the BC faces?)
3. **Residual ↔ error correlation** — does high Laplacian residual predict high prediction error per node?
4. **Multi-case comparison** — same material across interpolation / extrapolation-above Q values
5. **Summary table** — mean |∇²T̂| and MAE per test case

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import torch
import meshio
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from arch import ThermalNet

ELMER_CASES = ROOT / 'elmer_cases' / 'thermal_ccx_beam'
SAVES       = ROOT / 'saves'
LOG_COLS    = [3]

MATERIAL_K = {
    'Steel_A36':        50.0,
    'Steel_S355':       48.0,
    'Aluminium_6061':  167.0,
    'Titanium_Ti6Al4V':  6.7,
    'Concrete_C30':      1.8,
}

print('ROOT:', ROOT)
print('PyTorch:', torch.__version__)

ROOT: /home/riccardo/Documents/AI_for_NumSim/01_beam_studies/Beam_FEM_Thermal_CCX
PyTorch: 2.7.1


## 1 — Load model and normalisation parameters

In [2]:
ckpt   = torch.load(SAVES / 'thermal_pinn.pt', map_location='cpu', weights_only=True)
model  = ThermalNet(hidden=ckpt['hidden'])
state  = ckpt['model_state']
if any(k.startswith('module.') for k in state):
    state = {k[len('module.'):]: v for k, v in state.items()}
model.load_state_dict(state)
model.eval()

norm   = dict(np.load(SAVES / 'norm_params.npz'))
X_mean = norm['X_mean']   # (6,)
X_std  = norm['X_std']    # (6,)
Y_mean = norm['Y_mean']   # (1,)
Y_std  = norm['Y_std']    # (1,)

# spatial std needed to convert normalised 2nd-deriv → physical units
sx = X_std[:3]   # std of x, y, z  [mm]

n_params = sum(p.numel() for p in model.parameters())
print(f'ThermalNet  hidden={ckpt["hidden"]}  params={n_params:,}')
print(f'X_mean = {X_mean}')
print(f'X_std  = {X_std}')
print(f'Y_mean = {Y_mean}°C   Y_std = {Y_std}°C')
print(f'sx (spatial std in mm): {sx}')

ThermalNet  hidden=512  params=792,065
X_mean = [499.1243    49.6726    49.686653   8.826892  52.344334  19.602581]
X_std  = [295.76358     34.0049      33.97192      1.5437715   58.89446
   0.39934832]
Y_mean = [243.454]°C   Y_std = [699.7877]°C
sx (spatial std in mm): [295.76358  34.0049   33.97192]


## 2 — Helper functions

### 2a — `compute_laplacian_field`
Returns the **per-node Laplacian residual** ∇²T̂ in normalised coordinates.

We use the same autograd trick as `losses.py` but:
- Return per-node values instead of a scalar mean
- Process nodes in small batches to keep memory in check
- Optionally scale to physical units (÷ sx[j]²)

In [3]:
def compute_laplacian_field(model, X_n: np.ndarray,
                            batch_size: int = 1024,
                            physical_units: bool = True) -> np.ndarray:
    """
    Compute ∇²T̂ at every node in X_n.

    Parameters
    ----------
    X_n          : (N, 6) normalised input array
    batch_size   : nodes processed per autograd call (trade speed vs memory)
    physical_units: if True, divide each H_jj by sx[j]² to get ∇²T in mm⁻² units

    Returns
    -------
    lap : (N,) float32  — signed Laplacian per node
    """
    sx_t = torch.tensor(sx, dtype=torch.float32)   # (3,)
    all_lap = []

    for start in range(0, len(X_n), batch_size):
        xb = torch.from_numpy(X_n[start:start + batch_size])  # (B, 6)

        with torch.enable_grad():
            x  = xb.clone().requires_grad_(True)
            T  = model(x)                                              # (B, 1)

            # first spatial derivatives: g[i, j] = ∂T̂_i / ∂x̂_j
            g  = torch.autograd.grad(
                    T.sum(), x, create_graph=True)[0][:, :3]          # (B, 3)

            lap = torch.zeros(len(xb))
            for j in range(3):
                # second derivative diagonal: ∂²T̂_i / ∂x̂_j²
                H_jj = torch.autograd.grad(
                    g[:, j].sum(), x,
                    retain_graph=(j < 2),   # keep graph for next j
                    create_graph=False
                )[0][:, j].detach()         # (B,)

                if physical_units:
                    H_jj = H_jj / (sx_t[j] ** 2)
                lap = lap + H_jj

        all_lap.append(lap.numpy())

    return np.concatenate(all_lap).astype(np.float32)

### 2b — `load_case`
Loads a case from disk and returns everything needed for analysis.

In [4]:
def normalise_X(X: np.ndarray) -> np.ndarray:
    X_p = X.copy()
    for c in LOG_COLS:
        X_p[:, c] = np.log(X_p[:, c])
    return ((X_p - X_mean) / X_std).astype(np.float32)


def load_case(case_id: str) -> dict:
    """
    Load one case and return a dict with:
      coords  (N, 3)  — x, y, z in mm
      true_C  (N,)    — FEM temperature [°C]
      X_n     (N, 6)  — normalised feature matrix
      pred_C  (N,)    — model prediction [°C]
      params  dict    — case metadata
    """
    case_dir    = ELMER_CASES / case_id
    params      = json.loads((case_dir / 'case_params.json').read_text())
    mesh        = meshio.read(str(case_dir / 'case.vtk'))
    coords      = mesh.points.astype(np.float32)                       # (N, 3)

    temp_key    = next(k for k in mesh.point_data if k.lower() == 'temperature')
    true_C      = mesh.point_data[temp_key].squeeze().astype(np.float32)

    k_val       = MATERIAL_K.get(params['material'], params.get('k_mW_mm_C', 1.0))
    N           = len(coords)
    case_feats  = np.tile(
        np.array([params['q_total_mW'], k_val, params['T_fix_C']], dtype=np.float32),
        (N, 1)
    )
    X           = np.hstack([coords, case_feats])                      # (N, 6)
    X_n         = normalise_X(X)

    with torch.no_grad():
        Y_n     = model(torch.from_numpy(X_n)).numpy()                 # (N, 1)
    pred_C      = (Y_n * Y_std + Y_mean).squeeze().astype(np.float32)

    return dict(coords=coords, true_C=true_C, X_n=X_n, pred_C=pred_C,
                params=params, k_val=k_val)


print('Helpers defined.')

Helpers defined.


## 3 — Single-case deep dive

We pick one **interpolation** test case (Steel_A36, Q=52.5 W) to start.
The beam spans x=[0, 1000] mm, cross-section y=[0,100] z=[0,100] mm.
- x=0 face: Dirichlet BC (T = T_fix)
- x=1000 face: Neumann BC (heat flux q)

In [5]:
CASE_ID = 'case_0102'   # Steel_A36, Q=52.5 W, interpolation

print(f'Loading {CASE_ID} …')
case = load_case(CASE_ID)
params = case['params']

print(f"  Material : {params['material']}  (k = {case['k_val']} W/m/K)")
print(f"  Q_total  : {params['q_total_mW']/1000:.2f} W")
print(f"  T_fix    : {params['T_fix_C']:.1f} °C")
print(f"  N nodes  : {len(case['coords']):,}")
print(f"  FEM T range : {case['true_C'].min():.2f} – {case['true_C'].max():.2f} °C")
print(f"  pred T range: {case['pred_C'].min():.2f} – {case['pred_C'].max():.2f} °C")

print('\nComputing Laplacian field …')
lap = compute_laplacian_field(model, case['X_n'])
print(f'  |∇²T̂| stats: mean={np.abs(lap).mean():.4e}  max={np.abs(lap).max():.4e}')

Loading case_0102 …
  Material : Steel_A36  (k = 50.0 W/m/K)
  Q_total  : 52.50 W
  T_fix    : 20.0 °C
  N nodes  : 9,314
  FEM T range : 20.00 – 192.00 °C
  pred T range: 34.06 – 201.12 °C

Computing Laplacian field …
  |∇²T̂| stats: mean=4.4794e-06  max=1.2819e-05


In [6]:
coords  = case['coords']
true_C  = case['true_C']
pred_C  = case['pred_C']
err     = np.abs(pred_C - true_C)
abs_lap = np.abs(lap)

x, y, z = coords[:, 0], coords[:, 1], coords[:, 2]
vmin_T, vmax_T = true_C.min(), true_C.max()

fig = plt.figure(figsize=(20, 5))
fig.suptitle(
    f"{CASE_ID}  |  {params['material']}  |  Q={params['q_total_mW']/1000:.1f} W  "
    f"[{params['split']}]",
    fontsize=12
)

panels = [
    (true_C,   'FEM temperature',         'plasma',  vmin_T,          vmax_T,           'T [°C]'),
    (pred_C,   'ThermalNet prediction',   'plasma',  vmin_T,          vmax_T,           'T [°C]'),
    (err,      'Absolute error |ΔT|',     'Reds',    0,               err.max(),        '|ΔT| [°C]'),
    (abs_lap,  '|∇²T̂|  (Laplacian res.)','YlOrRd',  0,               np.percentile(abs_lap, 99), '|∇²T̂|'),
]

for col, (data, title, cmap, vlo, vhi, label) in enumerate(panels):
    ax = fig.add_subplot(1, 4, col + 1, projection='3d')
    sc = ax.scatter(x, y, z, c=data, cmap=cmap, vmin=vlo, vmax=vhi, s=3, alpha=0.7)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('X [mm]', fontsize=7)
    ax.set_ylabel('Y [mm]', fontsize=7)
    ax.set_zlabel('Z [mm]', fontsize=7)
    plt.colorbar(sc, ax=ax, shrink=0.55, label=label)

plt.tight_layout()
out = SAVES / f'laplacian_4panel_{CASE_ID}.png'
plt.savefig(out, dpi=130)
plt.show()
print(f'Saved → {out.relative_to(ROOT)}')

Saved → saves/laplacian_4panel_case_0102.png


/tmp/ipykernel_88928/3448850646.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4 — Boundary profile: |∇²T̂| vs x-position

For steady-state 1D heat conduction the exact solution is **linear in x** → ∇²T = 0 everywhere.
We expect the residual to be near-zero in the bulk and potentially elevated near:
- **x = 0** (Dirichlet BC: T = T_fix)
- **x = 1000 mm** (Neumann BC: heat flux)

If the model violates PDE more at the faces, that's a sign the BCs are not perfectly learnt.

In [7]:
# Bin nodes by x position and compute mean/max |∇²T̂| and |ΔT| per bin
N_BINS = 40
x_edges = np.linspace(x.min(), x.max(), N_BINS + 1)
x_centres = 0.5 * (x_edges[:-1] + x_edges[1:])

bin_idx = np.digitize(x, x_edges) - 1
bin_idx = np.clip(bin_idx, 0, N_BINS - 1)

mean_lap  = np.array([abs_lap[bin_idx == b].mean() if (bin_idx == b).any() else 0 for b in range(N_BINS)])
max_lap   = np.array([abs_lap[bin_idx == b].max()  if (bin_idx == b).any() else 0 for b in range(N_BINS)])
mean_err  = np.array([err[bin_idx == b].mean()     if (bin_idx == b).any() else 0 for b in range(N_BINS)])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle(f'Laplacian residual profile along beam axis — {CASE_ID}', fontsize=12)

ax = axes[0]
ax.fill_between(x_centres, mean_lap, alpha=0.4, color='darkorange', label='mean |∇²T̂|')
ax.plot(x_centres, mean_lap, color='darkorange', linewidth=1.5)
ax.plot(x_centres, max_lap,  color='red',        linewidth=1, linestyle='--', label='max |∇²T̂|')
ax.axvline(0,    color='navy', linestyle=':', linewidth=1.5, label='Dirichlet BC (x=0)')
ax.axvline(1000, color='firebrick', linestyle=':', linewidth=1.5, label='Neumann BC (x=L)')
ax.set_xlabel('x [mm]', fontsize=10)
ax.set_ylabel('|∇²T̂|', fontsize=10)
ax.set_title('Physics residual along x', fontsize=10)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.fill_between(x_centres, mean_err, alpha=0.4, color='steelblue', label='mean |ΔT| [°C]')
ax2.plot(x_centres, mean_err, color='steelblue', linewidth=1.5)
ax2.axvline(0,    color='navy', linestyle=':', linewidth=1.5, label='Dirichlet BC (x=0)')
ax2.axvline(1000, color='firebrick', linestyle=':', linewidth=1.5, label='Neumann BC (x=L)')
ax2.set_xlabel('x [mm]', fontsize=10)
ax2.set_ylabel('|ΔT| [°C]', fontsize=10)
ax2.set_title('Prediction error along x', fontsize=10)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
out = SAVES / f'laplacian_profile_{CASE_ID}.png'
plt.savefig(out, dpi=130)
plt.show()
print(f'Saved → {out.relative_to(ROOT)}')

Saved → saves/laplacian_profile_case_0102.png


/tmp/ipykernel_88928/3074801405.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5 — Correlation: does high |∇²T̂| predict high |ΔT|?

If the physics loss is meaningful, nodes where the Laplacian residual is large should also tend to have larger prediction errors. We check this with a scatter plot and Spearman rank correlation.

In [8]:
from scipy.stats import spearmanr

rho, pval = spearmanr(abs_lap, err)
print(f'Spearman ρ(|∇²T̂|, |ΔT|) = {rho:.3f}   (p = {pval:.2e})')

# subsample for readable scatter
rng = np.random.default_rng(0)
idx = rng.choice(len(abs_lap), size=min(3000, len(abs_lap)), replace=False)

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(abs_lap[idx], err[idx], c=coords[idx, 0],
                cmap='coolwarm', s=8, alpha=0.6)
plt.colorbar(sc, ax=ax, label='x position [mm]')
ax.set_xlabel('|∇²T̂|  (Laplacian residual)', fontsize=10)
ax.set_ylabel('|ΔT|  (prediction error) [°C]', fontsize=10)
ax.set_title(
    f'{CASE_ID}  |  Spearman ρ = {rho:.3f}\n'
    f'Colour = x position (blue=Dirichlet face, red=Neumann face)',
    fontsize=10
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = SAVES / f'laplacian_corr_{CASE_ID}.png'
plt.savefig(out, dpi=130)
plt.show()
print(f'Saved → {out.relative_to(ROOT)}')

Spearman ρ(|∇²T̂|, |ΔT|) = 0.514   (p = 0.00e+00)
Saved → saves/laplacian_corr_case_0102.png


/tmp/ipykernel_88928/1219954241.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6 — Multi-case comparison: interpolation vs extrapolation

We load all **Steel_A36** test cases (5 Q levels) to see how the Laplacian residual changes as Q moves from below the training range → training range → above training range.

| case_id   | Q [W] | regime         |
|-----------|-------|----------------|
| case_0100 | 0.1   | extrap-below   |
| case_0101 | 0.3   | extrap-below   |
| case_0102 | 52.5  | interp         |
| case_0103 | 200   | extrap-above   |
| case_0104 | 500   | extrap-above   |

In [9]:
STEEL_CASES = [
    ('case_0100',  0.1,    'extrap-below',  'C0'),
    ('case_0101',  0.3,    'extrap-below',  'C1'),
    ('case_0102',  52.5,   'interp',        'C2'),
    ('case_0103',  200.0,  'extrap-above',  'C3'),
    ('case_0104',  500.0,  'extrap-above',  'C4'),
]

results = []
for cid, q_W, regime, color in STEEL_CASES:
    print(f'  {cid}  Q={q_W} W  [{regime}] …')
    c = load_case(cid)
    lap_c = compute_laplacian_field(model, c['X_n'])
    err_c = np.abs(c['pred_C'] - c['true_C'])
    results.append(dict(
        case_id=cid, q_W=q_W, regime=regime, color=color,
        coords=c['coords'], abs_lap=np.abs(lap_c), err=err_c,
    ))
    print(f'    mean|∇²T̂|={np.abs(lap_c).mean():.4e}   MAE={err_c.mean():.3f} °C')

  case_0100  Q=0.1 W  [extrap-below] …
    mean|∇²T̂|=4.9212e-08   MAE=1.222 °C
  case_0101  Q=0.3 W  [extrap-below] …
    mean|∇²T̂|=1.4165e-07   MAE=0.329 °C
  case_0102  Q=52.5 W  [interp] …
    mean|∇²T̂|=4.4794e-06   MAE=4.377 °C
  case_0103  Q=200.0 W  [extrap-above] …
    mean|∇²T̂|=7.2315e-06   MAE=101.319 °C
  case_0104  Q=500.0 W  [extrap-above] …
    mean|∇²T̂|=5.0857e-06   MAE=469.919 °C


In [10]:
# Profile plot: mean |∇²T̂| vs x-position for all 5 cases
N_BINS = 40

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Steel_A36 — physics residual and prediction error along beam axis', fontsize=12)

for ax, key, ylabel in zip(
    axes,
    ['abs_lap', 'err'],
    ['mean |∇²T̂|  (physics residual)', 'mean |ΔT| [°C]  (prediction error)'],
):
    for r in results:
        xs = r['coords'][:, 0]
        x_edges = np.linspace(xs.min(), xs.max(), N_BINS + 1)
        x_centres = 0.5 * (x_edges[:-1] + x_edges[1:])
        bin_idx = np.clip(np.digitize(xs, x_edges) - 1, 0, N_BINS - 1)
        vals = r[key]
        mean_v = np.array([vals[bin_idx == b].mean() if (bin_idx == b).any() else 0
                           for b in range(N_BINS)])
        ax.plot(x_centres, mean_v, color=r['color'], linewidth=1.8,
                label=f"{r['q_W']} W [{r['regime']}]")

    ax.axvspan(0,    30,   alpha=0.08, color='navy',    label='Dirichlet face')
    ax.axvspan(970, 1000,  alpha=0.08, color='firebrick', label='Neumann face')
    ax.set_xlabel('x [mm]', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
out = SAVES / 'laplacian_multicase_steel.png'
plt.savefig(out, dpi=130)
plt.show()
print(f'Saved → {out.relative_to(ROOT)}')

Saved → saves/laplacian_multicase_steel.png


/tmp/ipykernel_88928/2831738609.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# Distribution of |∇²T̂| across the 5 cases (log-scale boxplot + strip)
fig, ax = plt.subplots(figsize=(9, 5))

labels = [f"{r['q_W']} W\n[{r['regime']}]" for r in results]
data   = [r['abs_lap'] for r in results]
colors = [r['color']   for r in results]

bp = ax.boxplot(data, labels=labels, patch_artist=True,
                flierprops=dict(marker='.', markersize=2, alpha=0.3),
                showfliers=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)

ax.set_yscale('log')
ax.set_ylabel('|∇²T̂|  (log scale)', fontsize=10)
ax.set_title('Distribution of per-node Laplacian residual — Steel_A36 test cases', fontsize=11)
ax.grid(True, which='both', alpha=0.3, axis='y')

plt.tight_layout()
out = SAVES / 'laplacian_boxplot_steel.png'
plt.savefig(out, dpi=130)
plt.show()
print(f'Saved → {out.relative_to(ROOT)}')

/tmp/ipykernel_88928/352085630.py:8: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data, labels=labels, patch_artist=True,


Saved → saves/laplacian_boxplot_steel.png


/tmp/ipykernel_88928/352085630.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7 — Summary table: all 25 test cases

Compute mean |∇²T̂| and MAE for every test case and print a comparison table.
This lets us see if the Laplacian residual tracks prediction error across materials and Q levels.

In [12]:
manifest   = json.loads((ELMER_CASES / 'vtk_manifest.json').read_text())
test_cases = [e for e in manifest['cases'] if e.get('split') == 'test' and e.get('success')]

summary_rows = []
for entry in test_cases:
    cid = entry['case_id']
    print(f'  {cid} …', end=' ', flush=True)
    c     = load_case(cid)
    lap_c = compute_laplacian_field(model, c['X_n'])
    err_c = np.abs(c['pred_C'] - c['true_C'])

    q_mW = c['params']['q_total_mW']
    if q_mW < 500:
        regime = 'extrap-below'
    elif q_mW > 100_000:
        regime = 'extrap-above'
    else:
        regime = 'interp'

    summary_rows.append(dict(
        case_id  = cid,
        material = c['params']['material'],
        q_W      = q_mW / 1000,
        regime   = regime,
        mean_lap = float(np.abs(lap_c).mean()),
        max_lap  = float(np.abs(lap_c).max()),
        p99_lap  = float(np.percentile(np.abs(lap_c), 99)),
        mae      = float(err_c.mean()),
    ))
    print(f"mean|∇²T̂|={summary_rows[-1]['mean_lap']:.3e}  MAE={summary_rows[-1]['mae']:.3f} °C")

print(f'\nDone — {len(summary_rows)} cases.')

  case_0100 … mean|∇²T̂|=4.921e-08  MAE=1.222 °C
  case_0101 … mean|∇²T̂|=1.416e-07  MAE=0.329 °C
  case_0102 … mean|∇²T̂|=4.479e-06  MAE=4.377 °C
  case_0103 … mean|∇²T̂|=7.231e-06  MAE=101.319 °C
  case_0104 … mean|∇²T̂|=5.086e-06  MAE=469.919 °C
  case_0205 … mean|∇²T̂|=4.582e-08  MAE=1.268 °C
  case_0206 … mean|∇²T̂|=1.295e-07  MAE=0.341 °C
  case_0207 … mean|∇²T̂|=4.385e-06  MAE=4.221 °C
  case_0208 … mean|∇²T̂|=6.719e-06  MAE=108.297 °C
  case_0209 … mean|∇²T̂|=4.382e-06  MAE=493.595 °C
  case_0310 … mean|∇²T̂|=1.901e-07  MAE=1.444 °C
  case_0311 … mean|∇²T̂|=2.215e-07  MAE=1.782 °C
  case_0312 … mean|∇²T̂|=1.288e-06  MAE=3.192 °C
  case_0313 … mean|∇²T̂|=3.083e-06  MAE=36.571 °C
  case_0314 … mean|∇²T̂|=5.645e-06  MAE=139.827 °C
  case_0415 … mean|∇²T̂|=1.437e-07  MAE=4.294 °C
  case_0416 … mean|∇²T̂|=2.472e-07  MAE=3.570 °C
  case_0417 … mean|∇²T̂|=1.520e-05  MAE=4.528 °C
  case_0418 … mean|∇²T̂|=9.725e-05  MAE=681.634 °C
  case_0419 … mean|∇²T̂|=1.712e-04  MAE=1402.784 °C
  ca

In [13]:
# Print formatted table
hdr = f"{'Case':12s}  {'Material':22s}  {'Q [W]':>8}  {'Regime':14s}  "\
      f"{'mean|∇²T̂|':>12}  {'p99|∇²T̂|':>12}  {'MAE [°C]':>10}"
print(hdr)
print('-' * len(hdr))

MAT_ORDER = list(MATERIAL_K.keys())
for mat in MAT_ORDER:
    sub = sorted([r for r in summary_rows if r['material'] == mat], key=lambda r: r['q_W'])
    for r in sub:
        print(f"{r['case_id']:12s}  {r['material']:22s}  {r['q_W']:8.3g}  "
              f"{r['regime']:14s}  {r['mean_lap']:12.3e}  {r['p99_lap']:12.3e}  {r['mae']:10.3f}")
    print()

Case          Material                   Q [W]  Regime            mean|∇²T̂|     p99|∇²T̂|    MAE [°C]
------------------------------------------------------------------------------------------------------
case_0100     Steel_A36                    0.1  extrap-below       4.921e-08     1.499e-07       1.222
case_0101     Steel_A36                    0.3  extrap-below       1.416e-07     3.167e-07       0.329
case_0102     Steel_A36                   52.5  interp             4.479e-06     1.190e-05       4.377
case_0103     Steel_A36                    200  extrap-above       7.231e-06     1.340e-05     101.319
case_0104     Steel_A36                    500  extrap-above       5.086e-06     9.050e-06     469.919

case_0205     Steel_S355                   0.1  extrap-below       4.582e-08     1.446e-07       1.268
case_0206     Steel_S355                   0.3  extrap-below       1.295e-07     3.078e-07       0.341
case_0207     Steel_S355                  52.5  interp             4.385

In [14]:
# Scatter: mean |∇²T̂| vs MAE, coloured by material, marker by regime
REGIME_MARKERS = {'extrap-below': 'v', 'interp': 'o', 'extrap-above': '^'}
COLORS = {m: plt.cm.tab10.colors[i] for i, m in enumerate(MAT_ORDER)}

fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle('Physics residual vs prediction error — all 25 test cases', fontsize=12)

for mat in MAT_ORDER:
    for regime, mkr in REGIME_MARKERS.items():
        sub = [r for r in summary_rows if r['material'] == mat and r['regime'] == regime]
        if not sub:
            continue
        ax.scatter(
            [r['mean_lap'] for r in sub],
            [r['mae']      for r in sub],
            color=COLORS[mat], marker=mkr, s=90, zorder=3,
            label=f'{mat} [{regime}]' if regime == 'interp' else '_',
        )
        for r in sub:
            ax.annotate(f"{r['q_W']:.4g}W",
                        (r['mean_lap'], r['mae']),
                        fontsize=6, alpha=0.7,
                        xytext=(4, 2), textcoords='offset points')

# legend: materials (colour) + regimes (marker)
from matplotlib.lines import Line2D
leg_mat = [Line2D([0], [0], color=COLORS[m], marker='o', linestyle='', label=m)
           for m in MAT_ORDER]
leg_reg = [Line2D([0], [0], color='grey', marker=mkr, linestyle='', label=regime)
           for regime, mkr in REGIME_MARKERS.items()]
ax.legend(handles=leg_mat + leg_reg, fontsize=7, ncol=2, loc='upper left')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('mean |∇²T̂|  (per-case physics residual)', fontsize=10)
ax.set_ylabel('MAE [°C]  (prediction error)', fontsize=10)
ax.grid(True, which='both', linestyle='--', alpha=0.4)

plt.tight_layout()
out = SAVES / 'laplacian_vs_mae_scatter.png'
plt.savefig(out, dpi=130)
plt.show()
print(f'Saved → {out.relative_to(ROOT)}')

Saved → saves/laplacian_vs_mae_scatter.png


/tmp/ipykernel_88928/380999212.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Aggregate stats by regime
from scipy.stats import spearmanr

print('\n=== Aggregate by regime ===')
for regime in ['interp', 'extrap-below', 'extrap-above']:
    sub = [r for r in summary_rows if r['regime'] == regime]
    if not sub:
        continue
    laps = [r['mean_lap'] for r in sub]
    maes = [r['mae']      for r in sub]
    print(f'\n[{regime}]  n={len(sub)}')
    print(f'  mean|∇²T̂| : {np.mean(laps):.3e}  (range: {min(laps):.3e} – {max(laps):.3e})')
    print(f'  MAE       : {np.mean(maes):.2f} °C  (range: {min(maes):.2f} – {max(maes):.2f})')

all_laps = [r['mean_lap'] for r in summary_rows]
all_maes = [r['mae']      for r in summary_rows]
rho, p = spearmanr(all_laps, all_maes)
print(f'\nSpearman ρ(mean|∇²T̂|, MAE) across all 25 cases = {rho:.3f}  (p={p:.3e})')


=== Aggregate by regime ===

[interp]  n=5
  mean|∇²T̂| : 1.099e-05  (range: 1.288e-06 – 2.961e-05)
  MAE       : 6.08 °C  (range: 3.19 – 14.08)

[extrap-below]  n=10
  mean|∇²T̂| : 2.620e-07  (range: 4.582e-08 – 1.011e-06)
  MAE       : 2.41 °C  (range: 0.33 – 5.01)

[extrap-above]  n=10
  mean|∇²T̂| : 6.104e-05  (range: 3.083e-06 – 1.712e-04)
  MAE       : 2186.74 °C  (range: 36.57 – 16071.85)

Spearman ρ(mean|∇²T̂|, MAE) across all 25 cases = 0.867  (p=2.079e-08)


---
## Key questions to read from the plots

| Question | Where to look |
|---|---|
| Does the model satisfy ∇²T=0 in the bulk? | 4-panel plot, panel 4 — should be near-zero away from faces |
| Do BC faces have higher residual? | Profile plot — look for peaks at x=0 and x=1000 |
| Does Laplacian residual predict error per-node? | Correlation scatter + Spearman ρ |
| Does residual grow with extrapolation? | Multi-case profile + boxplot |
| Is there a per-case correlation between physics residual and MAE? | Summary scatter (log-log) |

If `ρ(mean|∇²T̂|, MAE)` is high → the physics residual is a useful **unsupervised error indicator** (no FEM needed to estimate model quality).